# Тренажер: розгортка призми

Розгортка — це плоске зображення поверхні многогранника, отримане суміщенням його граней з однією площиною без зміни довжин.

Модель описує **правильну пряму призму** трьома параметрами:
- `n` — кількість сторін правильної основи;
- `s` — довжина сторони основи;
- `H` — висота призми.

Бічна поверхня розгортається у смугу з `n` прямокутників шириною `s` і висотою `H`. Дві правильні основи приєднуються до смуги.

**Ідея NO-CAD:** студент задає параметри геометричного об'єкта, а розміри й розгортка будуються алгоритмічно.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider, Checkbox
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

def regular_polygon_from_side(n, s):
    """Правильний n-кутник із першою стороною на осі X."""
    pts = [np.array([0.0, 0.0]), np.array([s, 0.0])]
    direction = 0.0
    turn = np.pi - 2*np.pi/n
    for _ in range(n-2):
        direction += turn
        pts.append(pts[-1] + s*np.array([np.cos(direction), np.sin(direction)]))
    return np.array(pts)

def prism_net(n=5, s=2.0, H=4.0, show_labels=True):
    base = regular_polygon_from_side(n, s)
    perimeter = n*s
    center = base.mean(axis=0)
    base3 = base - center
    lower = np.column_stack([base3[:, 0], base3[:, 1], np.zeros(n)])
    upper = lower.copy()
    upper[:, 2] = H

    fig = plt.figure(figsize=(14, 6.5))
    ax3d = fig.add_subplot(121, projection="3d")
    axnet = fig.add_subplot(122)

    faces = []
    for i in range(n):
        j = (i+1) % n
        quad = [lower[i], lower[j], upper[j], upper[i]]
        faces.append(quad)
        ax3d.plot([lower[i,0], lower[j,0], upper[j,0], upper[i,0], lower[i,0]],
                  [lower[i,1], lower[j,1], upper[j,1], upper[i,1], lower[i,1]],
                  [lower[i,2], lower[j,2], upper[j,2], upper[i,2], lower[i,2]],
                  color="steelblue", lw=1.3)
    faces.append([lower[i] for i in range(n)])
    faces.append([upper[i] for i in range(n)])
    ax3d.add_collection3d(Poly3DCollection(faces, alpha=0.12, facecolor="cornflowerblue", edgecolor="none"))
    lo = np.vstack([lower, lower[0]])
    up = np.vstack([upper, upper[0]])
    ax3d.plot(lo[:,0], lo[:,1], lo[:,2], color="steelblue", lw=1.8)
    ax3d.plot(up[:,0], up[:,1], up[:,2], color="steelblue", lw=1.8)
    ax3d.set_title("Просторова модель призми")
    ax3d.set_xlabel("X"); ax3d.set_ylabel("Y"); ax3d.set_zlabel("Z")
    span = max(s*n/(2*np.pi), H, s, 1)
    ax3d.set_box_aspect((1, 1, max(H/(2*span), .5)))
    ax3d.view_init(elev=22, azim=-55)
    ax3d.set_xlim(-span, span); ax3d.set_ylim(-span, span); ax3d.set_zlim(0, H*1.05)

    # Бічна поверхня: n прямокутників шириною s та висотою H
    for i in range(n):
        x0, x1 = i*s, (i+1)*s
        axnet.plot([x0,x1,x1,x0,x0], [0,0,H,H,0], color="black", lw=1.4)
        if show_labels:
            axnet.text((x0+x1)/2, H/2, f"Б{i+1}", ha="center", va="center", fontsize=10)
            axnet.text((x0+x1)/2, -0.15, f"{s:g}", ha="center", va="top", fontsize=8)
    axnet.plot([0, perimeter, perimeter, 0, 0], [0,0,H,H,0], color="black", lw=1.8)

    # Дві основи приєднано до першої бічної грані.
    top_poly = base.copy(); top_poly[:,1] += H
    bottom_poly = base.copy(); bottom_poly[:,1] *= -1
    for poly in (top_poly, bottom_poly):
        closed = np.vstack([poly, poly[0]])
        axnet.plot(closed[:,0], closed[:,1], color="darkorange", lw=1.8)
    axnet.plot([0,s], [H,H], color="darkorange", lw=2.8)
    axnet.plot([0,s], [0,0], color="darkorange", lw=2.8)

    if show_labels:
        offset = max(s,H)*0.12
        axnet.text(s/2, H+offset, "Основа 1", ha="center", va="bottom", fontsize=9)
        axnet.text(s/2, -offset, "Основа 2", ha="center", va="top", fontsize=9)
        axnet.text(perimeter/2, H+max(s,H)*0.30, f"P = n·s = {perimeter:g}",
                   ha="center", va="bottom", fontsize=10)
        axnet.text(perimeter/2, H/2, f"Площа бічної поверхні = {perimeter*H:g}",
                   ha="center", va="center", fontsize=9, alpha=.75)

    margin = max(s,H)*0.55
    axnet.set_xlim(-margin, perimeter+margin)
    axnet.set_ylim(-max(s,H)*0.65, H+max(s,H)*0.65)
    axnet.set_aspect("equal", adjustable="box")
    axnet.grid(True, alpha=.2)
    axnet.set_xlabel("Розмір у площині розгортки")
    axnet.set_ylabel("Розмір у площині розгортки")
    axnet.set_title("Розгортка поверхні призми")
    fig.suptitle(f"Правильна {n}-кутна пряма призма: s = {s:g}, H = {H:g}", fontsize=14)
    plt.tight_layout()
    plt.show()

    print(f"Кількість бічних граней: {n}")
    print(f"Периметр основи: P = n·s = {n}·{s:g} = {perimeter:g}")
    print(f"Площа бічної поверхні: Sбіч = P·H = {perimeter:g}·{H:g} = {perimeter*H:g}")
    print(f"Розгортка бічної поверхні: ширина {perimeter:g}, висота {H:g}")

interact(
    prism_net,
    n=IntSlider(min=3, max=10, step=1, value=5, description="Сторін n"),
    s=FloatSlider(min=0.5, max=4.0, step=0.25, value=2.0, description="Сторона s"),
    H=FloatSlider(min=1.0, max=7.0, step=0.5, value=4.0, description="Висота H"),
    show_labels=Checkbox(value=True, description="Підписи")
)


## Практичні завдання

1. Встановіть `n = 3`. Дослідіть розгортку трикутної призми.
2. Змініть кількість сторін від 3 до 10. Які елементи розгортки змінюються?
3. Збільшіть `s`, не змінюючи `n` та `H`. Простежте за зміною ширини розгортки.
4. Збільшіть `H`. Які розміри залишаються незмінними?
5. Для заданих параметрів вручну обчисліть периметр основи та площу бічної поверхні, а потім перевірте результат у тренажері.

## Що має зрозуміти студент

- периметр правильної основи: `P = n·s`;
- бічна поверхня складається з `n` прямокутників шириною `s` і висотою `H`;
- ширина смуги дорівнює периметру основи, а висота — висоті призми;
- відповідні ребра основ і бічних граней мають однакову довжину.

**Наступний крок:** побудувати розгортку довільної призми, де студент задає координати вершин основи, а програма обчислює довжини ребер і розміри бічних граней.
